In [7]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
BACKEND_PATH = PROJECT_ROOT / "backend"

sys.path.insert(0, str(BACKEND_PATH))

print("Project root:", PROJECT_ROOT)
print("Backend path:", BACKEND_PATH)

Project root: e:\DataSentinel
Backend path: e:\DataSentinel\backend


In [8]:
import app

print("app imported successfully")

app imported successfully


In [10]:
from app.ingestion.loaders import load_dataset

df = load_dataset("../data/raw/Online Retail.xlsx")

print("Shape:", df.shape)

Shape: (541909, 8)


In [15]:
from app.quality.types import validate_data_types

type_results = validate_data_types(df)

type_results

[{'check': 'data_type_validation',
  'column': 'InvoiceNo',
  'detected_dtype': 'object',
  'issue': 'mixed_data_types',
  'severity': 'medium'},
 {'check': 'data_type_validation',
  'column': 'StockCode',
  'detected_dtype': 'object',
  'issue': 'mixed_data_types',
  'severity': 'medium'},
 {'check': 'data_type_validation',
  'column': 'Description',
  'detected_dtype': 'object',
  'issue': 'mixed_data_types',
  'severity': 'medium'}]

In [12]:
for column in ["InvoiceNo", "StockCode", "Description"]:
    print(f"\n{column}")
    print(df[column].dropna().map(type).value_counts())


InvoiceNo
InvoiceNo
<class 'int'>    532618
<class 'str'>      9291
Name: count, dtype: int64

StockCode
StockCode
<class 'int'>    487036
<class 'str'>     54873
Name: count, dtype: int64

Description
Description
<class 'str'>    540454
<class 'int'>         1
Name: count, dtype: int64


In [13]:
df[df["Description"].apply(lambda x: isinstance(x, int))]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
420391,572891,23343,20713,-400,2011-10-26 14:14:00,0.0,NaN,United Kingdom


In [14]:
df[df["Description"].apply(lambda x: isinstance(x, int))]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
420391,572891,23343,20713,-400,2011-10-26 14:14:00,0.0,NaN,United Kingdom


In [16]:
from app.quality.types import validate_data_types

type_results = validate_data_types(df)

type_results

[{'check': 'data_type_validation',
  'column': 'InvoiceNo',
  'detected_dtype': 'object',
  'issue': 'mixed_data_types',
  'severity': 'medium'},
 {'check': 'data_type_validation',
  'column': 'StockCode',
  'detected_dtype': 'object',
  'issue': 'mixed_data_types',
  'severity': 'medium'},
 {'check': 'data_type_validation',
  'column': 'Description',
  'detected_dtype': 'object',
  'issue': 'mixed_data_types',
  'severity': 'medium'}]

In [17]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
BACKEND_PATH = PROJECT_ROOT / "backend"

if str(BACKEND_PATH) not in sys.path:
    sys.path.insert(0, str(BACKEND_PATH))

print(BACKEND_PATH)

e:\DataSentinel\backend


In [18]:
from app.ingestion.loaders import load_dataset

df = load_dataset("../data/raw/Online Retail.xlsx")

print(df.shape)

(541909, 8)


In [19]:
from app.quality.formats import validate_column_format

country_result = validate_column_format(
    df=df,
    column="Country",
    pattern=r"[A-Za-z ]+",
    format_name="alphabetic_country_name",
)

country_result

{'check': 'format_validation',
 'column': 'Country',
 'format': 'alphabetic_country_name',
 'total_values': 541909,
 'invalid_count': 0,
 'invalid_percentage': 0.0,
 'severity': 'none'}

In [20]:
invoice_result = validate_column_format(
    df=df,
    column="InvoiceNo",
    pattern=r"\d+",
    format_name="numeric_invoice_number",
)

invoice_result

{'check': 'format_validation',
 'column': 'InvoiceNo',
 'format': 'numeric_invoice_number',
 'total_values': 541909,
 'invalid_count': 9291,
 'invalid_percentage': 1.71,
 'severity': 'low'}

In [21]:
country_result

{'check': 'format_validation',
 'column': 'Country',
 'format': 'alphabetic_country_name',
 'total_values': 541909,
 'invalid_count': 0,
 'invalid_percentage': 0.0,
 'severity': 'none'}

In [22]:
invoice_result

{'check': 'format_validation',
 'column': 'InvoiceNo',
 'format': 'numeric_invoice_number',
 'total_values': 541909,
 'invalid_count': 9291,
 'invalid_percentage': 1.71,
 'severity': 'low'}

In [23]:
df["Country"].value_counts().head(20)

Country
United Kingdom     495478
Germany              9495
France               8557
EIRE                 8196
Spain                2533
Netherlands          2371
Belgium              2069
Switzerland          2002
Portugal             1519
Australia            1259
Norway               1086
Italy                 803
Channel Islands       758
Finland               695
Cyprus                622
Sweden                462
Unspecified           446
Austria               401
Denmark               389
Japan                 358
Name: count, dtype: int64

In [24]:
from app.quality.consistency import detect_category_inconsistency

country_consistency = detect_category_inconsistency(
    df,
    "Country"
)

country_consistency

{'check': 'category_consistency',
 'column': 'Country',
 'inconsistent_groups': 0,
 'affected_values': 0,
 'severity': 'none'}

In [25]:
test_df = pd.DataFrame({
    "Country": [
        "United Kingdom",
        "United Kingdom",
        "united kingdom",
        "UNITED KINGDOM",
        "France",
        "France",
        "Germany"
    ]
})

In [26]:
test_result = detect_category_inconsistency(
    test_df,
    "Country"
)

test_result

{'check': 'category_consistency',
 'column': 'Country',
 'inconsistent_groups': 1,
 'affected_values': 3,
 'severity': 'low'}

In [27]:
country_consistency

{'check': 'category_consistency',
 'column': 'Country',
 'inconsistent_groups': 0,
 'affected_values': 0,
 'severity': 'none'}

In [28]:
test_result

{'check': 'category_consistency',
 'column': 'Country',
 'inconsistent_groups': 1,
 'affected_values': 3,
 'severity': 'low'}

In [29]:
print("Negative quantities:")
print((df["Quantity"] < 0).sum())

print("\nZero quantities:")
print((df["Quantity"] == 0).sum())

print("\nNegative prices:")
print((df["UnitPrice"] < 0).sum())

print("\nZero prices:")
print((df["UnitPrice"] == 0).sum())

Negative quantities:
10624

Zero quantities:
0

Negative prices:
2

Zero prices:
2515


In [30]:
df[df["Quantity"] < 0][
    ["InvoiceNo", "StockCode", "Description", "Quantity", "UnitPrice"]
].head(10)

,InvoiceNo,StockCode,Description,Quantity,UnitPrice
141,C536379,D,Discount,-1,27.50
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,4.65
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,1.65
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,0.29
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,0.29
238,C536391,21980,PACK OF 12 RED RETROSPOT TISSUES,-24,0.29
239,C536391,21484,CHICK GREY HOT WATER BOTTLE,-12,3.45
240,C536391,22557,PLASTERS IN TIN VINTAGE PAISLEY,-12,1.65
241,C536391,22553,PLASTERS IN TIN SKULLS,-24,1.65
939,C536506,22960,JAM MAKING SET WITH JARS,-6,4.25


In [31]:
from app.quality.constraints import validate_numeric_constraints

price_result = validate_numeric_constraints(
    df=df,
    column="UnitPrice",
    min_value=0,
    allow_zero=True,
)

price_result

{'check': 'numeric_constraint',
 'column': 'UnitPrice',
 'min_value': 0,
 'max_value': None,
 'allow_zero': True,
 'invalid_count': 2,
 'invalid_percentage': 0.0,
 'severity': 'low'}

In [32]:
quantity_result = validate_numeric_constraints(
    df=df,
    column="Quantity",
    min_value=0,
    allow_zero=False,
)

quantity_result

{'check': 'numeric_constraint',
 'column': 'Quantity',
 'min_value': 0,
 'max_value': None,
 'allow_zero': False,
 'invalid_count': 10624,
 'invalid_percentage': 1.96,
 'severity': 'low'}

In [33]:
quantity_result = validate_numeric_constraints(
    df=df,
    column="Quantity",
    min_value=0,
    allow_zero=False,
)

quantity_result

{'check': 'numeric_constraint',
 'column': 'Quantity',
 'min_value': 0,
 'max_value': None,
 'allow_zero': False,
 'invalid_count': 10624,
 'invalid_percentage': 1.96,
 'severity': 'low'}

In [34]:
from app.quality.outliers import detect_iqr_outliers

price_outliers = detect_iqr_outliers(
    df=df,
    column="UnitPrice",
)

price_outliers

{'check': 'iqr_outlier',
 'column': 'UnitPrice',
 'q1': 1.25,
 'q3': 4.13,
 'iqr': 2.88,
 'lower_bound': -3.07,
 'upper_bound': 8.45,
 'outlier_count': 39627,
 'outlier_percentage': 7.31,
 'severity': 'medium'}

In [35]:
quantity_outliers = detect_iqr_outliers(
    df=df,
    column="Quantity",
)

quantity_outliers

{'check': 'iqr_outlier',
 'column': 'Quantity',
 'q1': 1.0,
 'q3': 10.0,
 'iqr': 9.0,
 'lower_bound': -12.5,
 'upper_bound': 23.5,
 'outlier_count': 58619,
 'outlier_percentage': 10.82,
 'severity': 'high'}

In [36]:
from app.quality.engine import run_quality_checks

quality_results = run_quality_checks(df)

quality_results

{'missing_values': [{'check': 'missing_values',
   'column': 'Description',
   'missing_count': 1454,
   'missing_percentage': 0.27,
   'severity': 'low'},
  {'check': 'missing_values',
   'column': 'CustomerID',
   'missing_count': 135080,
   'missing_percentage': 24.93,
   'severity': 'critical'}],
 'duplicate_rows': {'check': 'duplicate_rows',
  'duplicate_count': 10147,
  'duplicate_percentage': 1.87,
  'severity': 'medium'},
 'data_type_validation': [{'check': 'data_type_validation',
   'column': 'InvoiceNo',
   'detected_dtype': 'object',
   'issue': 'mixed_data_types',
   'severity': 'medium'},
  {'check': 'data_type_validation',
   'column': 'StockCode',
   'detected_dtype': 'object',
   'issue': 'mixed_data_types',
   'severity': 'medium'},
  {'check': 'data_type_validation',
   'column': 'Description',
   'detected_dtype': 'object',
   'issue': 'mixed_data_types',
   'severity': 'medium'}],
 'outliers': [{'check': 'iqr_outlier',
   'column': 'Quantity',
   'q1': 1.0,
   'q3':

In [37]:
quality_results["missing_values"]

[{'check': 'missing_values',
  'column': 'Description',
  'missing_count': 1454,
  'missing_percentage': 0.27,
  'severity': 'low'},
 {'check': 'missing_values',
  'column': 'CustomerID',
  'missing_count': 135080,
  'missing_percentage': 24.93,
  'severity': 'critical'}]

In [38]:
quality_results["duplicate_rows"]

{'check': 'duplicate_rows',
 'duplicate_count': 10147,
 'duplicate_percentage': 1.87,
 'severity': 'medium'}

In [39]:
quality_results["data_type_validation"]

[{'check': 'data_type_validation',
  'column': 'InvoiceNo',
  'detected_dtype': 'object',
  'issue': 'mixed_data_types',
  'severity': 'medium'},
 {'check': 'data_type_validation',
  'column': 'StockCode',
  'detected_dtype': 'object',
  'issue': 'mixed_data_types',
  'severity': 'medium'},
 {'check': 'data_type_validation',
  'column': 'Description',
  'detected_dtype': 'object',
  'issue': 'mixed_data_types',
  'severity': 'medium'}]

In [40]:
quality_results["outliers"]

[{'check': 'iqr_outlier',
  'column': 'Quantity',
  'q1': 1.0,
  'q3': 10.0,
  'iqr': 9.0,
  'lower_bound': -12.5,
  'upper_bound': 23.5,
  'outlier_count': 58619,
  'outlier_percentage': 10.82,
  'severity': 'high'},
 {'check': 'iqr_outlier',
  'column': 'UnitPrice',
  'q1': 1.25,
  'q3': 4.13,
  'iqr': 2.88,
  'lower_bound': -3.07,
  'upper_bound': 8.45,
  'outlier_count': 39627,
  'outlier_percentage': 7.31,
  'severity': 'medium'},
 {'check': 'iqr_outlier',
  'column': 'CustomerID',
  'q1': 13953.0,
  'q3': 16791.0,
  'iqr': 2838.0,
  'lower_bound': 9696.0,
  'upper_bound': 21048.0,
  'outlier_count': 0,
  'outlier_percentage': 0.0,
  'severity': 'none'}]

In [41]:
from app.quality.engine import run_quality_checks

quality_results = run_quality_checks(df)

quality_results

{'missing_values': [{'check': 'missing_values',
   'column': 'Description',
   'missing_count': 1454,
   'missing_percentage': 0.27,
   'severity': 'low'},
  {'check': 'missing_values',
   'column': 'CustomerID',
   'missing_count': 135080,
   'missing_percentage': 24.93,
   'severity': 'critical'}],
 'duplicate_rows': {'check': 'duplicate_rows',
  'duplicate_count': 10147,
  'duplicate_percentage': 1.87,
  'severity': 'medium'},
 'data_type_validation': [{'check': 'data_type_validation',
   'column': 'InvoiceNo',
   'detected_dtype': 'object',
   'issue': 'mixed_data_types',
   'severity': 'medium'},
  {'check': 'data_type_validation',
   'column': 'StockCode',
   'detected_dtype': 'object',
   'issue': 'mixed_data_types',
   'severity': 'medium'},
  {'check': 'data_type_validation',
   'column': 'Description',
   'detected_dtype': 'object',
   'issue': 'mixed_data_types',
   'severity': 'medium'}],
 'outliers': [{'check': 'iqr_outlier',
   'column': 'Quantity',
   'q1': 1.0,
   'q3':

In [42]:
quality_results["missing_values"]

[{'check': 'missing_values',
  'column': 'Description',
  'missing_count': 1454,
  'missing_percentage': 0.27,
  'severity': 'low'},
 {'check': 'missing_values',
  'column': 'CustomerID',
  'missing_count': 135080,
  'missing_percentage': 24.93,
  'severity': 'critical'}]

In [43]:
quality_results["duplicate_rows"]

{'check': 'duplicate_rows',
 'duplicate_count': 10147,
 'duplicate_percentage': 1.87,
 'severity': 'medium'}

In [44]:
quality_results["data_type_validation"]

[{'check': 'data_type_validation',
  'column': 'InvoiceNo',
  'detected_dtype': 'object',
  'issue': 'mixed_data_types',
  'severity': 'medium'},
 {'check': 'data_type_validation',
  'column': 'StockCode',
  'detected_dtype': 'object',
  'issue': 'mixed_data_types',
  'severity': 'medium'},
 {'check': 'data_type_validation',
  'column': 'Description',
  'detected_dtype': 'object',
  'issue': 'mixed_data_types',
  'severity': 'medium'}]

In [45]:
quality_results["outliers"]

[{'check': 'iqr_outlier',
  'column': 'Quantity',
  'q1': 1.0,
  'q3': 10.0,
  'iqr': 9.0,
  'lower_bound': -12.5,
  'upper_bound': 23.5,
  'outlier_count': 58619,
  'outlier_percentage': 10.82,
  'severity': 'high'},
 {'check': 'iqr_outlier',
  'column': 'UnitPrice',
  'q1': 1.25,
  'q3': 4.13,
  'iqr': 2.88,
  'lower_bound': -3.07,
  'upper_bound': 8.45,
  'outlier_count': 39627,
  'outlier_percentage': 7.31,
  'severity': 'medium'},
 {'check': 'iqr_outlier',
  'column': 'CustomerID',
  'q1': 13953.0,
  'q3': 16791.0,
  'iqr': 2838.0,
  'lower_bound': 9696.0,
  'upper_bound': 21048.0,
  'outlier_count': 0,
  'outlier_percentage': 0.0,
  'severity': 'none'}]

In [46]:
from app.findings.generator import generate_findings

findings = generate_findings(quality_results)

findings

[{'check': 'missing_values',
  'column': 'Description',
  'issue': 'missing_values',
  'severity': 'low',
  'count': 1454,
  'percentage': 0.27,
  'details': {'check': 'missing_values',
   'column': 'Description',
   'missing_count': 1454,
   'missing_percentage': 0.27,
   'severity': 'low'}},
 {'check': 'missing_values',
  'column': 'CustomerID',
  'issue': 'missing_values',
  'severity': 'critical',
  'count': 135080,
  'percentage': 24.93,
  'details': {'check': 'missing_values',
   'column': 'CustomerID',
   'missing_count': 135080,
   'missing_percentage': 24.93,
   'severity': 'critical'}},
 {'check': 'duplicate_rows',
  'column': None,
  'issue': 'duplicate_rows',
  'severity': 'medium',
  'count': 10147,
  'percentage': 1.87,
  'details': {'check': 'duplicate_rows',
   'duplicate_count': 10147,
   'duplicate_percentage': 1.87,
   'severity': 'medium'}},
 {'check': 'data_type_validation',
  'column': 'InvoiceNo',
  'issue': 'mixed_data_types',
  'severity': 'medium',
  'count': 

In [47]:
len(findings)

8

In [48]:
for finding in findings:
    print(
        finding["severity"].upper(),
        "|",
        finding["check"],
        "|",
        finding["column"],
        "|",
        finding["count"],
    )

LOW | missing_values | Description | 1454
CRITICAL | missing_values | CustomerID | 135080
MEDIUM | duplicate_rows | None | 10147
MEDIUM | data_type_validation | InvoiceNo | None
MEDIUM | data_type_validation | StockCode | None
MEDIUM | data_type_validation | Description | None
HIGH | iqr_outlier | Quantity | 58619
MEDIUM | iqr_outlier | UnitPrice | 39627


In [50]:
from app.scoring.quality_score import calculate_quality_score

quality_score = calculate_quality_score(findings)

quality_score

{'score': 43,
 'grade': 'poor',
 'total_findings': 8,
 'total_penalty': 57,
 'severity_counts': {'critical': 1, 'high': 1, 'medium': 5, 'low': 1}}

In [51]:
print("Score:", quality_score["score"])
print("Grade:", quality_score["grade"])
print("Findings:", quality_score["total_findings"])
print("Penalty:", quality_score["total_penalty"])
print("Severity:", quality_score["severity_counts"])

Score: 43
Grade: poor
Findings: 8
Penalty: 57
Severity: {'critical': 1, 'high': 1, 'medium': 5, 'low': 1}


In [52]:
from app.rules.rule_engine import apply_business_rules

business_rules = apply_business_rules(df)

business_rules

[{'rule': 'legitimate_return_quantity',
  'columns': ['InvoiceNo', 'Quantity'],
  'classification': 'expected_business_behavior',
  'affected_count': 9288,
  'message': 'Negative quantities associated with cancellation invoices are treated as legitimate returns/cancellations.'}]

In [53]:
for rule in business_rules:
    print("Rule:", rule["rule"])
    print("Classification:", rule["classification"])
    print("Affected rows:", rule["affected_count"])
    print("Message:", rule["message"])

Rule: legitimate_return_quantity
Classification: expected_business_behavior
Affected rows: 9288
Message: Negative quantities associated with cancellation invoices are treated as legitimate returns/cancellations.


In [54]:
from app.repair.recommendations import generate_recommendations

recommendations = generate_recommendations(
    findings=findings,
    business_rules=business_rules,
)

recommendations

[{'check': 'missing_values',
  'column': 'Description',
  'severity': 'low',
  'action': 'investigate',
  'reason': 'Small amounts of missing data may be handled depending on column importance.',
  'safe_to_auto_repair': False},
 {'check': 'missing_values',
  'column': 'CustomerID',
  'severity': 'critical',
  'action': 'human_review',
  'reason': 'High-impact missing values require business review before modification.',
  'safe_to_auto_repair': False},
 {'check': 'duplicate_rows',
  'column': None,
  'severity': 'medium',
  'action': 'review_duplicates',
  'reason': 'Duplicate rows should be reviewed before removal because repeated transactions may sometimes be legitimate.',
  'safe_to_auto_repair': False},
 {'check': 'data_type_validation',
  'column': 'InvoiceNo',
  'severity': 'medium',
  'action': 'standardize_type',
  'reason': 'Mixed data types should be standardized after verifying the intended column type.',
  'safe_to_auto_repair': False},
 {'check': 'data_type_validation',
 

In [55]:
for recommendation in recommendations:
    print(
        recommendation["action"],
        "|",
        recommendation["column"],
        "|",
        recommendation["reason"],
    )

investigate | Description | Small amounts of missing data may be handled depending on column importance.
human_review | CustomerID | High-impact missing values require business review before modification.
review_duplicates | None | Duplicate rows should be reviewed before removal because repeated transactions may sometimes be legitimate.
standardize_type | InvoiceNo | Mixed data types should be standardized after verifying the intended column type.
standardize_type | StockCode | Mixed data types should be standardized after verifying the intended column type.
standardize_type | Description | Mixed data types should be standardized after verifying the intended column type.
investigate_outliers | Quantity | Statistical outliers are not automatically errors and should be investigated before changing their values.
investigate_outliers | UnitPrice | Statistical outliers are not automatically errors and should be investigated before changing their values.
keep_value | Quantity | Negative qua

In [56]:
from app.repair.repair_engine import repair_duplicate_rows

cleaned_df, repair_result = repair_duplicate_rows(
    df=df,
    approved=False,
)

repair_result

{'repair': 'duplicate_rows',
 'approved': False,
 'changed': False,
 'removed_count': 0,
 'message': 'Duplicate removal was not approved. Original dataset remains unchanged.'}

In [57]:
print("Original:", len(df))
print("Cleaned:", len(cleaned_df))

Original: 541909
Cleaned: 541909


In [58]:
cleaned_df, repair_result = repair_duplicate_rows(
    df=df,
    approved=True,
)

repair_result

{'repair': 'duplicate_rows',
 'approved': True,
 'changed': True,
 'removed_count': 5268,
 'message': 'Removed 5268 completely duplicated rows.'}

In [59]:
print("Original rows:", len(df))
print("Cleaned rows:", len(cleaned_df))
print("Duplicates after repair:", cleaned_df.duplicated().sum())

Original rows: 541909
Cleaned rows: 536641
Duplicates after repair: 0


In [60]:
from app.repair.audit import create_audit_record

audit_record = create_audit_record(
    repair_result=repair_result,
    before_rows=len(df),
    after_rows=len(cleaned_df),
)

audit_record

{'timestamp': '2026-09-03T11:46:21.985492',
 'repair': 'duplicate_rows',
 'approved': True,
 'changed': True,
 'before_rows': 541909,
 'after_rows': 536641,
 'removed_count': 5268,
 'message': 'Removed 5268 completely duplicated rows.'}

In [61]:
for key, value in audit_record.items():
    print(f"{key}: {value}")

timestamp: 2026-09-03T11:46:21.985492
repair: duplicate_rows
approved: True
changed: True
before_rows: 541909
after_rows: 536641
removed_count: 5268
message: Removed 5268 completely duplicated rows.


In [62]:
from app.validation.validator import validate_repair

validation_result = validate_repair(
    original_df=df,
    repaired_df=cleaned_df,
)

validation_result

{'validation_passed': True,
 'original_rows': 541909,
 'repaired_rows': 536641,
 'rows_removed': 5268,
 'original_duplicates': 5268,
 'repaired_duplicates': 0,
 'duplicate_reduction': 5268}

In [63]:
for key, value in validation_result.items():
    print(f"{key}: {value}")

validation_passed: True
original_rows: 541909
repaired_rows: 536641
rows_removed: 5268
original_duplicates: 5268
repaired_duplicates: 0
duplicate_reduction: 5268


In [64]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
BACKEND_PATH = PROJECT_ROOT / "backend"

if str(BACKEND_PATH) not in sys.path:
    sys.path.insert(0, str(BACKEND_PATH))

In [65]:
from app.ingestion.loaders import load_dataset

df = load_dataset("../data/raw/Online Retail.xlsx")

print(df.shape)

(541909, 8)


In [66]:
from app.quality.types import validate_data_types

type_results = validate_data_types(df)

for result in type_results:
    print(result)

{'check': 'data_type_validation', 'column': 'InvoiceNo', 'detected_dtype': 'object', 'issue': 'mixed_data_types', 'severity': 'medium'}
{'check': 'data_type_validation', 'column': 'StockCode', 'detected_dtype': 'object', 'issue': 'mixed_data_types', 'severity': 'medium'}
{'check': 'data_type_validation', 'column': 'Description', 'detected_dtype': 'object', 'issue': 'mixed_data_types', 'severity': 'medium'}


In [67]:
from app.quality.types import validate_data_types

type_results = validate_data_types(df)

for result in type_results:
    print(result)

{'check': 'data_type_validation', 'column': 'InvoiceNo', 'detected_dtype': 'object', 'issue': 'mixed_data_types', 'severity': 'medium'}
{'check': 'data_type_validation', 'column': 'StockCode', 'detected_dtype': 'object', 'issue': 'mixed_data_types', 'severity': 'medium'}
{'check': 'data_type_validation', 'column': 'Description', 'detected_dtype': 'object', 'issue': 'mixed_data_types', 'severity': 'medium'}


In [68]:
import app.quality.types as types_module

print("Loaded file:")
print(types_module.__file__)

print("\nFunction source:")
import inspect
print(inspect.getsource(types_module.validate_data_types))

Loaded file:
e:\DataSentinel\backend\app\quality\types.py

Function source:
def validate_data_types(df: pd.DataFrame) -> list[dict]:
    """
    Detect columns containing mixed Python data types.

    The dominant Python type is treated as the expected type.
    Values belonging to other types are reported as unexpected.
    """

    findings = []

    for column in df.columns:
        series = df[column].dropna()

        if series.empty:
            continue

        type_counts = series.map(
            lambda value: type(value).__name__
        ).value_counts()

        if len(type_counts) <= 1:
            continue

        dominant_type = type_counts.index[0]
        unexpected_types = type_counts.index[1:]

        unexpected_count = int(type_counts.iloc[1:].sum())
        total_values = len(series)

        unexpected_percentage = (
            unexpected_count / total_values
        ) * 100

        if unexpected_percentage >= 10:
            severity = "high"
        elif une

In [69]:
import app.quality.types as types_module
import inspect

print("LOADED FILE:")
print(types_module.__file__)

print("\nACTUAL FUNCTION:")
print(inspect.getsource(types_module.validate_data_types))

LOADED FILE:
e:\DataSentinel\backend\app\quality\types.py

ACTUAL FUNCTION:
def validate_data_types(df: pd.DataFrame) -> list[dict]:
    """
    Detect columns containing mixed Python data types.

    The dominant Python type is treated as the expected type.
    Values belonging to other types are reported as unexpected.
    """

    findings = []

    for column in df.columns:
        series = df[column].dropna()

        if series.empty:
            continue

        type_counts = series.map(
            lambda value: type(value).__name__
        ).value_counts()

        if len(type_counts) <= 1:
            continue

        dominant_type = type_counts.index[0]
        unexpected_types = type_counts.index[1:]

        unexpected_count = int(type_counts.iloc[1:].sum())
        total_values = len(series)

        unexpected_percentage = (
            unexpected_count / total_values
        ) * 100

        if unexpected_percentage >= 10:
            severity = "high"
        elif une

In [70]:
import importlib
import app.quality.types as types_module

importlib.reload(types_module)

<module 'app.quality.types' from 'e:\\DataSentinel\\backend\\app\\quality\\types.py'>

In [71]:
type_results = types_module.validate_data_types(df)

for result in type_results:
    print(result)

{'check': 'data_type_validation', 'column': 'InvoiceNo', 'dominant_type': 'int', 'unexpected_types': ['str'], 'unexpected_count': 9291, 'unexpected_percentage': 1.7145, 'severity': 'medium'}
{'check': 'data_type_validation', 'column': 'StockCode', 'dominant_type': 'int', 'unexpected_types': ['str'], 'unexpected_count': 54873, 'unexpected_percentage': 10.1259, 'severity': 'high'}
{'check': 'data_type_validation', 'column': 'Description', 'dominant_type': 'str', 'unexpected_types': ['int'], 'unexpected_count': 1, 'unexpected_percentage': 0.0002, 'severity': 'low'}


In [72]:
import importlib
import app.quality.engine as engine_module

importlib.reload(engine_module)

quality_results = engine_module.run_quality_checks(df)

In [73]:
for result in quality_results["data_type_validation"]:
    print(result)

{'check': 'data_type_validation', 'column': 'InvoiceNo', 'dominant_type': 'int', 'unexpected_types': ['str'], 'unexpected_count': 9291, 'unexpected_percentage': 1.7145, 'severity': 'medium'}
{'check': 'data_type_validation', 'column': 'StockCode', 'dominant_type': 'int', 'unexpected_types': ['str'], 'unexpected_count': 54873, 'unexpected_percentage': 10.1259, 'severity': 'high'}
{'check': 'data_type_validation', 'column': 'Description', 'dominant_type': 'str', 'unexpected_types': ['int'], 'unexpected_count': 1, 'unexpected_percentage': 0.0002, 'severity': 'low'}


In [74]:
from app.findings.generator import generate_findings

findings = generate_findings(quality_results)

for finding in findings:
    print(
        finding["severity"],
        "|",
        finding["check"],
        "|",
        finding["column"],
        "|",
        finding["count"],
        "|",
        finding["percentage"]
    )

low | missing_values | Description | 1454 | 0.27
critical | missing_values | CustomerID | 135080 | 24.93
medium | duplicate_rows | None | 10147 | 1.87
medium | data_type_validation | InvoiceNo | 9291 | 1.7145
high | data_type_validation | StockCode | 54873 | 10.1259
low | data_type_validation | Description | 1 | 0.0002
high | iqr_outlier | Quantity | 58619 | 10.82
medium | iqr_outlier | UnitPrice | 39627 | 7.31


In [75]:
from app.scoring.quality_score import calculate_quality_score

quality_score = calculate_quality_score(findings)

quality_score

{'score': 41,
 'grade': 'poor',
 'total_findings': 8,
 'total_penalty': 59,
 'severity_counts': {'critical': 1, 'high': 2, 'medium': 3, 'low': 2}}

In [76]:
for key, value in quality_score.items():
    print(f"{key}: {value}")

score: 41
grade: poor
total_findings: 8
total_penalty: 59
severity_counts: {'critical': 1, 'high': 2, 'medium': 3, 'low': 2}


In [77]:
from app.quality.anomaly import detect_anomalies

In [78]:
quantity_anomalies = detect_anomalies(
    df,
    "Quantity",
)

quantity_anomalies

{'check': 'isolation_forest_anomaly',
 'column': 'Quantity',
 'anomaly_count': 4898,
 'anomaly_percentage': 0.9,
 'contamination': 0.01,
 'severity': 'low'}

In [79]:
price_anomalies = detect_anomalies(
    df,
    "UnitPrice",
)

price_anomalies

{'check': 'isolation_forest_anomaly',
 'column': 'UnitPrice',
 'anomaly_count': 4783,
 'anomaly_percentage': 0.88,
 'contamination': 0.01,
 'severity': 'low'}